# Anchoring regression V5 (LEC) - past *and* future lags

Per-neuron regression of raw 25 ms firing onto (location x goal-progress x lag) anchors,
leave-one-session-out, after El-Gaby et al. 2024 Figure 5. Module:
[`elasticnet_regression_v5.py`](elasticnet_regression_v5.py); v4 is left untouched, and
`elasticnet_v5_synthetics.py` control 8b asserts that V5 under the v4 flags reproduces v4 to
`max|diff| = 0`. The PFC mirror is
[`PFC_elasticnet_regression_v5.ipynb`](../mFC_data/code/PFC_elasticnet_regression_v5.ipynb).

**What V5 changes over V4** (see [`ELASTICNET_V5.md`](ELASTICNET_V5.md) and
[`ELGABY_FIGURE5_RECONCILIATION.md`](ELGABY_FIGURE5_RECONCILIATION.md))

El-Gaby's deposited code differs from the paper's text in three places that V4 had implemented
from the text. V5's defaults follow the code, because the code is what produced the figures:

| | paper text (V4) | his code (V5 default) |
|---|---|---|
| state-tuning statistic | peak per state and trial (`'max'`, leg-duration confounded) | **mean over the neuron's preferred-phase bins** (`'pref_phase_mean'`), NaN propagating |
| preferred phase | argmax of raw time-weighted mean rate (`'raw_mean'`) | **peak bin within each third** of the trial-averaged normalised curve (`'elgaby_peak'`) -- disagrees on 37% of LEC neuron-sessions |
| "with non-zero beta coefficients" | mean r finite | finite r in **every** fold |
| non-zero-lag neuron | majority of folds pass, all folds averaged | **>= 1 fold passes**, mean over passing folds |

Both semantics are reported side by side (`three_panel_summary`). Also new: session skip reasons
(`sessions_skipped`, in `run_config.json`), the reduced-beta r everywhere a non-zero-lag neuron
is scored, polar rate maps per held-out task (`*_polar.pdf`), and both Poisson readouts when
`use_poisson=True`.

**Two configs in this notebook**

* **science** (the LEC claim): `MIN_TRIALS = 5`, `pref_phase_source='train'` (no leakage),
  past and future lags. Selection by anatomy via `unit_regions.pkl`.
* **reproduction** (same criterion as the paper): `MIN_TRIALS = 1`, `pref_phase_source='test'`,
  past lags only. The companion to the PFC reproduction, so the two datasets can be compared
  under one criterion.

**Things to know before reading any output**

1. Anchor phase is *determined* by preferred phase and lag (`ap == (pref -/+ lag) % 3`), so
   only 108 of the 324 columns are live in any fit and the beta matrix is stored/plotted as
   `location x lag`.
2. The prediction is therefore **exactly zero** at every non-preferred-phase bin (the shaded
   thirds on the polar pages). Use `tuning_corr_pref`, not `tuning_corr`.
3. Preferred phase is refit per fold and a third of neurons change it, so betas are averaged
   only over folds sharing the modal phase; the `*_foldbetas.pdf` files show each fold.
4. At `alpha_mode='fixed', elasticnet_alpha=0.01` on raw counts, ~60% of neurons fit all-zero
   and the survivors are the fast-firing ones. `alpha_mode='relative'` puts every neuron at
   the same point on its own regularization path. Check `recday_diagnostics_*.csv`.
5. On this 3 Hz dataset his NaN-propagating state test removes 14.6% of neuron-sessions on
   its own and selects markedly higher-rate units; read region contrasts against firing rate.

Gate: `python elasticnet_v5_synthetics.py` - 57 controls, including exact reproduction of v3
and v4.

In [ ]:
import numpy as np
import scipy.stats as st
from scipy import stats
from scipy.stats import zscore
from scipy.ndimage import gaussian_filter1d
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os, pickle
from tqdm import tqdm

In [ ]:
Data_folder="../data/processed_data"
figures_folder="../data/figures"
neuron_folder = f"{Data_folder}/neuron_raw_mingyutest"
trialtimes_folder = f"{Data_folder}/trialtimes_raw_mingyutest"
tracking_folder = f"{Data_folder}/processed"

In [ ]:
import pickle
import os

# Define the folder where the processed data is saved
processed_data_folder = "../data/processed_data"

# Load the main data dictionary
with open(os.path.join(processed_data_folder, 'data_dic_lec.pkl'), 'rb') as f:
    data_dic = pickle.load(f)

# Load the normalized neurons dictionary
with open(os.path.join(processed_data_folder, 'norm_neurons_dic.pkl'), 'rb') as f:
    norm_neurons_dic = pickle.load(f)

# Load the session indices dictionary
with open(os.path.join(processed_data_folder, 'session_inds_dic.pkl'), 'rb') as f:
    session_inds_dic = pickle.load(f)

# Load the tasks dictionary
with open(os.path.join(processed_data_folder, 'tasks_dic.pkl'), 'rb') as f:
    tasks_dic = pickle.load(f)

print(f"Loaded data dictionaries from {processed_data_folder}")

In [ ]:
all_files = os.listdir(neuron_folder)
neuron_files = [f for f in all_files if f.startswith('Neuron_raw_') and f.endswith('.npy')]

mouse_recdays_ = []
for f in neuron_files:
    # Remove prefix 'Neuron_raw_' and suffix '.npy'
    stripped_name = f[len('Neuron_raw_'):-len('.npy')]
    # Split by '_' and rejoin all parts except the last one (session number)
    mouse_recday = '_'.join(stripped_name.split('_')[:-1])
    mouse_recdays_.append(mouse_recday)

# Find the unique mouse_recday identifiers
mouse_recdays = np.unique(mouse_recdays_)

# Filter out identifiers containing '_sb'
unique_mouse_recdays = [recday for recday in mouse_recdays if '_sb' not in recday]

print("Unique mouse_recday identifiers (without '_sb'):")
print(unique_mouse_recdays)

# Assign the first unique identifier to mouse_recday for later use
if len(unique_mouse_recdays) > 0:
    mouse_recday = unique_mouse_recdays[0]

mouse_recdays = unique_mouse_recdays

In [ ]:
# Session selection: one session per unique task (El-Gaby's `non_repeat_ses_maker`), for two
# trial thresholds. MIN_TRIALS_SCIENCE=5 is the LEC science run; MIN_TRIALS_REPRO=1 is his rule
# (num_trials > 0) for the same-criterion-as-the-paper companion run.
#
# Sessions that fail `_prepare_session` (no tracking, no completed trial) are reported per
# recday by the run itself and recorded in run_config.json; on this pickle that is 30 sessions:
# 21 Object-exploration blocks (no task) and 9 ABCD sessions in which the animal never completed
# a loop. A 31st (ah10_20250618_20250619 session 5, truncated camera pinstate) was salvaged on
# 2026-09-06 by code/preprocessing/salvage_tracking_offset.py and is now in the pickle; see
# ELASTICNET_V5.md and the lec-unusable-sessions-and-salvage memory note.
import importlib
import elasticnet_regression_v5 as v5
importlib.reload(v5)

MIN_TRIALS_SCIENCE = 5
MIN_TRIALS_REPRO = 1


def select_sessions(min_trials, verbose=True):
    out = {}
    for mouse_recday in mouse_recdays:
        valid_sessions, tasks = [], []
        for session in sorted(s for s in data_dic[mouse_recday] if s != 'valid_sessions'):
            sd = data_dic[mouse_recday][session]
            if 'defaultdict' in str(sd.get('Task')) or sd.get('Task') is None:
                if verbose:
                    print(f'{mouse_recday} session {session}: no task, skipping')
                continue
            if sd['num_trials'] < min_trials:
                if verbose:
                    print(f'{mouse_recday} session {session}: {sd["num_trials"]} trials '
                          f'< {min_trials}, skipping')
                continue
            if not any(np.array_equal(sd['Task'], c) for c in tasks):
                tasks.append(sd['Task'])
                valid_sessions.append(session)
        out[mouse_recday] = valid_sessions
    return v5.apply_excluded_sessions(out, verbose=False)   # inert on LEC (no me11 recday)


valid_sessions_dic = select_sessions(MIN_TRIALS_SCIENCE)
valid_sessions_dic_repro = select_sessions(MIN_TRIALS_REPRO, verbose=False)
for name, vsd in (('science', valid_sessions_dic), ('repro', valid_sessions_dic_repro)):
    n_fold = {mr: len(v) for mr, v in vsd.items()}
    print(f'\n[{name}] folds per recday: min {min(n_fold.values())}  median '
          f'{int(np.median(list(n_fold.values())))}  max {max(n_fold.values())}  '
          f'| recdays with <2 folds: {[mr for mr, n in n_fold.items() if n < 2] or "none"}')

## Run - past and future lags

Both directions, same config otherwise. `n_jobs` uses joblib's threading backend, so `data_dic` is shared rather than copied to workers.

In [ ]:
import importlib, os
from datetime import datetime
import elasticnet_regression_v5 as v5
importlib.reload(v5)

STAMP = datetime.now().strftime('%Y%m%d_%H%M%S')
N_JOBS = 6                      # 8 cores on this box; threading, so memory is shared
RUN_REPRO = True                # also run the same-criterion-as-the-paper companion (past only)
# 'none' = raw 25 ms spike counts, the reproduction. 'zscore_recday' divides each neuron's fit
# target by its sd over ALL used sessions of the recday (one scalar per neuron, so between-session
# rate differences within a recday survive), because a FIXED alpha=0.01 on raw counts is a
# firing-rate filter: 59% of neurons get an all-zero fit, 97% in the slowest rate quartile. Such
# runs are written to `*_zscore_v5_*` directories. The long runs go through
# `slurm_v5/submit_v5_run.sh` (job 8 = LEC past, 9 = LEC future), not this notebook.
Y_SCALING = 'none'


def make_config(direction, **kw):
    # Everything not named here comes from the V5 defaults, each read off El-Gaby's deposited
    # code (see ELGABY_FIGURE5_RECONCILIATION.md):
    #   state_tuning_statistic='pref_phase_mean'   his Figure2 cell 46 (paper text says 'max')
    #   state_tuning_nan_policy='propagate'        his scipy.stats.zscore + ttest_1samp
    #   pref_phase_method='elgaby_peak'            his tuning_phase_boolean_max (cell 31)
    #   nonzero_lag_zero_lags=(0, 11)              "lag from anchor of 30 degrees or more"
    #   state_tuning_min_fraction=1/3              "state-tuned in more than one-third of the tasks"
    # The SCIENCE run overrides pref_phase_source to 'train': the held-out session no longer
    # chooses which bins its own score averages over. Legacy (v4) behaviour is one call away:
    #   make_config(d, state_tuning_statistic='max', state_tuning_nan_policy='omit',
    #               pref_phase_method='raw_mean')
    params = dict(
        use_poisson=False, regularize=True,      # ElasticNet, alpha=0.01, positive
        lag_direction=direction,
        alpha_mode='fixed',                      # 'relative' + alpha_frac to rescale per neuron
        state_reduce='mean',                     # 'max' also stored either way
        pref_phase_source='train',               # science default; the repro run sets 'test'
        y_scaling=Y_SCALING,                     # 'zscore_recday' -> a `*_zscore_v5_*` run
    )
    params.update(kw)
    return v5.RegressionConfigV5(**params)


results, pooled, diagnostics = {}, {}, {}
for direction in ('past', 'future'):
    cfg = make_config(direction)
    out_dir = os.path.join('../data/figures', v5.run_dir_name(cfg, stamp=STAMP))
    print(f'\n{"="*70}\nSCIENCE  {direction.upper()} lags -> {out_dir}\n{"="*70}')
    results[direction], pooled[direction], diagnostics[direction] = v5.run_and_summarise_all_mice_v5(
        data_dic, cfg,
        valid_sessions_dic=valid_sessions_dic,
        save_dir=out_dir, export_dir=out_dir,
        make_pdfs=True, n_jobs=N_JOBS, verbose=True,
        manifest_extra={'min_trials': MIN_TRIALS_SCIENCE,
                        'run_kind': 'science' if Y_SCALING == 'none' else 'science_zscore',
                        'notebook': 'LEC_elasticnet_regression_v5'})
configs = {d: make_config(d) for d in results}

if RUN_REPRO:
    cfg_repro = make_config('past', pref_phase_source='test')
    out_repro = os.path.join('../data/figures', v5.run_dir_name(cfg_repro, stamp=STAMP,
                                                                prefix='repro_'))
    print(f'\n{"="*70}\nREPRODUCTION  PAST lags -> {out_repro}\n{"="*70}')
    results_repro, pooled_repro, diagnostics_repro = v5.run_and_summarise_all_mice_v5(
        data_dic, cfg_repro,
        valid_sessions_dic=valid_sessions_dic_repro,
        save_dir=out_repro, export_dir=out_repro,
        make_pdfs=False, n_jobs=N_JOBS, verbose=False,
        manifest_extra={'min_trials': MIN_TRIALS_REPRO, 'run_kind': 'reproduction',
                        'notebook': 'LEC_elasticnet_regression_v5'})

### Per-recday diagnostics

Read this before the region table. `frac_allzero_fits` is the alpha dropout, `frac_pref_phase_flips` the fraction of neurons whose coordinate frame changes between folds, and `n_nonzero_lag_alt_top3` the mask size with the argsort tie-breaking guard flipped - a large gap there means the mask is partly sort-order artefact.

In [ ]:
for direction, tab in diagnostics.items():
    print(f'\n===== {direction} =====')
    print(tab.to_string(index=False))

### The state-tuning filter: his statistic vs the paper's text

`selected = nonzero_lag & state_tuned & r.notna()`. V4 implemented the paper's text ("peak
firing rate in each state and trial", `state_tuning_statistic='max'`), which is **confounded
by leg duration**: `raw_to_norm` averages more raw bins into each normalised bin when a leg is
longer, lowering its variance and so its max, so constant-rate cells acquire the shortest leg
as their "preferred state" (measured FPR on pure noise: 0.05 at equal legs, 0.97 at 2x, 1.00 at
3x; this dataset's median longest:shortest ratio is 2.26x, and 47% of real units "prefer" the
shortest leg under `'max'` against a chance of 25%).

**El-Gaby's code does not use the peak.** Figure2 cell 46 takes the mean over the neuron's
preferred-phase bins per (trial, state) and lets NaN propagate through `zscore` and the t-test.
That statistic is not a max-of-averaged-bins and control 9 of the synthetics measures its FPR
at 3x legs near nominal. So "the reference has the confound too" was wrong: it applies to the
paper's text, not to what produced the figures. V5's default is his statistic; `'max'` is still
computed as `state_tuned_mask_alt` so the swap stays visible below.

On LEC his test passes a similar *number* of units to `'max'` (1,433 vs 1,477 of 2,851) but not
the same units, and the NaN propagation alone removes 14.6% of neuron-sessions -- it acts as a
firing-rate filter (tuned units 4-8 Hz vs untuned 0.5-2.5 Hz). Any region contrast below has to
be read against `mean_rate_hz`.

In [ ]:
# The tuning filter under his statistic vs the paper's 'max', the leg-duration ratio, and the
# skip bookkeeping. `n_selected_anyfold` / `n_tuned_finite_all_folds` are his semantics.
for direction, tab in diagnostics.items():
    print(f'\n===== {direction} =====')
    print(tab[['mouse_recday', 'n_neurons', 'n_folds', 'n_sessions_skipped', 'n_state_tuned',
               'n_state_tuned_alt_stat', 'n_tuned_finite_all_folds', 'n_selected',
               'n_selected_anyfold', 'state_duration_ratio',
               'frac_pref_state_is_shortest']].to_string(index=False))

# The same region table under the OTHER statistic ('max' when the run used his). If a region
# result only exists under one statistic, say so.
tab = tables['past'].copy()
tab['selected'] = tab.nonzero_lag & tab.state_tuned_alt_stat & tab.mean_corr_nonzero.notna()
print("\n===== past lags, state tuning by the OTHER statistic (state_tuned_mask_alt) =====")
v5.region_summary(tab)

## Which brain regions do these neurons come from?

In [ ]:
regions = v5.load_unit_regions()
tables = {d: v5.build_unit_table(results[d], configs[d], regions=regions, data_dic=data_dic)
          for d in results}
# how often his preferred-phase rule and the v4 raw-mean rule agree, per neuron (mean over sessions)
print('pref-phase agreement (elgaby_peak vs raw_mean):',
      {d: round(float(t.pref_phase_agreement.mean()), 3) for d, t in tables.items()})
for direction, tab in tables.items():
    print(f'\n{"="*70}\n{direction.upper()} lags - selection by region\n{"="*70}')
    v5.region_summary(tab)

### The paper's three panels, under both semantics

El-Gaby reports the same correlation under three selections: all state-tuned, non-zero-lag
(30 deg, excluding lags {0,11}) and strict non-zero-lag (90 deg, excluding {0,1,2,9,10,11}).
The `elgaby` block uses his semantics (pool = state-tuned with a finite r in every fold; a
non-zero-lag neuron needs >= 1 passing fold and is valued by the mean over passing folds); the
`v4` block is the per-neuron majority-vote mask with all folds averaged. The published PFC
values are printed for orientation only -- this is LEC, and the science run uses the
training-session preferred phase, which his analysis did not.

In [ ]:
# The paper's three panels, from the same fits, one block per (link, semantics).
panel_tables = {}
for direction, tab in tables.items():
    print(f'\n{"="*74}\nSCIENCE  {direction.upper()} lags\n{"="*74}')
    panel_tables[direction] = v5.three_panel_summary(tab)

if RUN_REPRO:
    table_repro = v5.build_unit_table(results_repro, cfg_repro, regions=regions, data_dic=data_dic)
    print(f'\n{"="*74}\nREPRODUCTION config (MIN_TRIALS=1, test-session preferred phase), PAST lags\n{"="*74}')
    panel_tables['repro_past'] = v5.three_panel_summary(table_repro)

### Past vs future, per region

`pro_index = (r_future - r_past) / (|r_future| + |r_past|)`: positive means a unit is better
explained prospectively. The two designs are not degenerate - past lag *k* and future lag
12-*k* point at the same task position one loop apart, and their measured column correlation
is only ~0.02-0.34, because routes vary between trials.

In [ ]:
merged, direction_summary = v4.compare_directions(tables['past'], tables['future'])
merged.head()

### The firing-rate confound

At a fixed ElasticNet alpha a unit is only fittable if it fires fast enough (all-zero fits
average ~0.7 Hz, surviving fits ~6.4 Hz), so a region difference in *selection rate* can be a
region difference in *firing rate*. `region_summary` already prints the within-quartile
version above. To check how much of the effect is the penalty rather than the anatomy, re-run
one recday with a per-neuron relative alpha and compare.

In [ ]:
# Rate-matched re-run of a single recday (cheap): every neuron sits at the same point on
# its own regularization path instead of a shared absolute alpha.
mr = list(results['past'].keys())[0]
cfg_rel = make_config('past', alpha_mode='relative', alpha_frac=0.1)
res_rel = v5.run_cross_validated_regression_v5(
    data_dic, mr, cfg_rel, valid_sessions=valid_sessions_dic[mr], verbose=True)
tab_rel = v5.build_unit_table({mr: res_rel}, cfg_rel, regions=regions, data_dic=data_dic)
print('\n--- relative alpha ---')
v5.region_summary(tab_rel)
print('\n--- fixed alpha, same recday ---')
v5.region_summary(tables['past'][tables['past'].recday == mr])

### The z-scored-target variant

`y_scaling='zscore_recday'` divides each neuron's fit target by its sd over the whole recday.
It is *not* part of the reproduction: it exists because the reference's fixed `alpha=0.01` on raw
25 ms counts is a firing-rate filter (the ElasticNet zeroing threshold scales with sd(y)), so the
selected population is partly a rate-selected population. Everything upstream of the fit -- the
state test, the preferred phases, the actual tuning curves -- is invariant to a positive
per-neuron affine transform and is bit-identical between the two runs (synthetic control 12), so
the comparison below isolates the effect of the penalty.

The cell reads the newest raw and `_zscore` run directories for one direction off disk (the runs
themselves are launched with `slurm_v5/submit_v5_run.sh`) and asks two questions: how many fits
stop being all-zero, and whether the newly admitted neurons are the low-spike ones.

In [ ]:
# Raw vs recday-z-scored target, from the stored runs (no re-fit).
import elasticnet_v5_compare as cmp5
importlib.reload(cmp5)
import numpy as np, pandas as pd, glob, os

DIRECTION = 'past'
FIG_ROOT = '../data/figures'


def _load_table(run_dir):
    """Rebuild a unit table from a run directory's npz exports."""
    res, cfg = {}, None
    for path in sorted(glob.glob(os.path.join(run_dir, f'*_{DIRECTION}_arrays.npz'))):
        z = v5.load_regression_outputs(path)
        rd = os.path.basename(path).replace(f'_{DIRECTION}_arrays.npz', '')
        res[rd] = z
        cfg = cfg or v5._config_from_results(z)
    return v5.build_unit_table(res, cfg, regions=regions, data_dic=data_dic, require_anatomy=False), cfg


pair = {}
for label, tag in (('raw counts', None), ('recday z-score', 'zscore')):
    try:
        run_dir = cmp5.resolve_latest(FIG_ROOT, DIRECTION, 'elasticnet', tag=tag)
    except FileNotFoundError as exc:
        print(f'{label}: {exc}')
        continue
    tab, cfg = _load_table(run_dir)
    pair[label] = (tab, cfg, run_dir)
    print(f'{label:16s} <- {os.path.basename(run_dir)}  ({len(tab)} neurons)')

for label, (tab, cfg, _) in pair.items():
    print(f'\n===== {label} =====')
    v5.three_panel_summary(tab)
    allzero = float((tab.n_nonzero_betas == 0).mean())
    print(f'  all-zero fits (mean over folds == 0): {allzero:.3f}')

# Did the extra neurons come from the low-spike end? Selection rate by spike-count quintile.
# `n_spikes_recday` was added with the y_scaling option, so a run older than that carries NaN
# there; the quintiles are taken from whichever table actually has it.
sizes = {lab: len(t) for lab, (t, _, _) in pair.items()}
ref = next((t for t, _, _ in pair.values() if np.isfinite(t.n_spikes_recday).any()), None)
if len(pair) == 2 and len(set(sizes.values())) > 1:
    print(f'\nrun sizes differ ({sizes}) -- not aligning neuron for neuron')
elif len(pair) == 2 and ref is None:
    print('\nneither run stores n_spikes_recday (both predate y_scaling); re-export to compare')
elif len(pair) == 2:
    rows = []
    q = pd.qcut(ref.n_spikes_recday, 5, labels=False, duplicates='drop')
    for label, (tab, _, _) in pair.items():
        for qi in sorted(pd.unique(q.dropna())):
            m = (q == qi).values
            rows.append({'target': label, 'spike_quintile': int(qi),
                          'median_spikes': float(np.nanmedian(tab.n_spikes_recday[m])),
                          'n': int(m.sum()),
                          'frac_allzero': float((tab.n_nonzero_betas[m] == 0).mean()),
                          'frac_state_tuned': float(tab.state_tuned[m].mean()),
                          'frac_nonzero_lag': float(tab.nonzero_lag[m].mean()),
                          'mean_r_nonzero': float(np.nanmean(tab.mean_corr_nonzero[m]))})
    admission = pd.DataFrame(rows).pivot(index='spike_quintile', columns='target')
    print('\n=== selection by recday spike-count quintile (the noise-admission caveat) ===')
    print(admission.to_string())

## Inspecting one neuron

`fold_betas` returns each fold's beta matrix collapsed in *that fold's* frame - the un-averaged view of what the `*_foldbetas.pdf` pages show.

In [ ]:
mr = list(results['past'].keys())[0]
res = results['past'][mr]
top = np.argsort(np.nan_to_num(res['mean_tuning_correlations_pref'], nan=-np.inf))[::-1][:5]
print('top neurons by preferred-phase tuning r:', top.tolist())
ni = int(top[0])
print(f'neuron {ni}: pref phase per fold = {res["pref_phases"][ni].tolist()}, '
      f'peak lag = {res["peak_lags"][ni]}, r = {res["mean_corrs"][ni]:.3f}, '
      f'reduced-beta r = {res["mean_corrs_nonzero"][ni]:.3f} '
      f'(passing folds only: {res["mean_corrs_nonzero_passing"][ni]:.3f}, '
      f'{int(res["n_passing_folds"][ni])} of {res["num_sessions"]}), '
      f'non-zero betas/fold = {res["n_nonzero_betas"][ni].tolist()}')
fb = v5.fold_betas(res, configs['past'], ni)          # (n_folds, 9 locations, 12 lags)
fig, axes = plt.subplots(1, len(fb) + 1, figsize=(2.2 * (len(fb) + 1), 2.4))
vmax = np.nanmax(np.abs(fb)) or 1.0
for fi, ax in enumerate(axes[:-1]):
    ax.imshow(fb[fi], aspect='auto', cmap='viridis', vmin=0, vmax=vmax)
    ax.set_title(f'fold {fi} (pref {res["pref_phases"][ni, fi]})', fontsize=7)
    ax.tick_params(labelsize=5)
B, modal, n_used, n_tot = v5.betas_in_common_frame(res, configs['past'], ni)
axes[-1].imshow(B, aspect='auto', cmap='viridis', vmin=0, vmax=vmax)
axes[-1].set_title(f'mean [{n_used}/{n_tot} folds, pref {modal}]', fontsize=7)
axes[-1].tick_params(labelsize=5)
fig.suptitle(f'{mr} neuron {ni} - past lags (location x lag)', fontsize=9)
fig.tight_layout()

# the same neuron as polar rate maps per held-out task (Fig 5g style), reduced-beta prediction solid
v5.plot_fold_polar_pages(res, configs['past'], f'../data/figures/{mr}_neuron{ni}_polar.pdf',
                         neuron_indices=[ni], prediction='nz')

## Reading the exports back

One `.npz` per recday per direction, plus eight PDFs:

| file | what it shows |
|---|---|
| `*_all.pdf` / `*_nonzerolag.pdf` | summary page per neuron: betas, 360-bin curves, n=4 readout |
| `*_all_foldbetas.pdf` / `*_nonzerolag_foldbetas.pdf` | beta matrix **per fold**, each in its own frame |
| `*_all_foldratemaps.pdf` / `*_nonzerolag_foldratemaps.pdf` | actual vs predicted **per fold**, linear axes |
| `*_all_polar.pdf` / `*_nonzerolag_polar.pdf` | actual vs predicted **per fold as polar rate maps** (Fig 5g style); on the `_nonzerolag` pages the solid predicted curve is the **reduced-beta** prediction, i.e. what that neuron's r is computed from |

A single fold's `r` comes from 4 points, whose null sampling SD is 1/sqrt(3) = 0.58: read the
shapes, not the individual numbers. Nothing below needs `data_dic`. `run_config.json` records
the config, `MIN_TRIALS`, the sessions used and skipped per recday (with reasons), and the
library versions.

In [ ]:
import glob, json
out_dir = os.path.join('../data/figures',
                       v5.run_dir_name(configs['past'], stamp=STAMP))
print('\n'.join(sorted(os.path.basename(p) for p in glob.glob(out_dir + '/*'))[:14]))
z = v5.load_regression_outputs(glob.glob(out_dir + '/*_arrays.npz')[0])
for k in sorted(z):
    print(f'  {k:40s} {getattr(z[k], "shape", type(z[k]).__name__)}')
man = json.load(open(os.path.join(out_dir, 'run_config.json')))
print('\nmanifest: min_trials =', man.get('min_trials'), '| run_kind =', man.get('run_kind'),
      '| sessions skipped in total =', man.get('sessions_skipped_total'))
skipped = {rd: v['sessions_skipped'] for rd, v in man['sessions_used'].items() if v['sessions_skipped']}
print('skipped sessions by recday:', json.dumps(skipped, indent=1)[:1500])